## import libraries

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

## Define Paths and Study Period

In [2]:
DATA_DIR = "electricity_dataset"
OUTPUT_DIR = "electricityEDA_outputs/reproducibility"

os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset_paths = sorted(Path(DATA_DIR).glob("*.csv"))

print("Number of files:", len(dataset_paths))

Number of files: 22


In [3]:
# Warm-up period is used only as historical reference for imputation and lag features.
warmup_start_date = pd.Timestamp("2015-07-01")

# Main modeling period starts here.
modeling_start_date = pd.Timestamp("2017-07-01")

# Train/validation/test split boundaries.
train_end_date = pd.Timestamp("2024-01-01")   # train: 2017H2 to 2023H2
valid_end_date = pd.Timestamp("2025-01-01")   # validation: 2024
test_end_date = pd.Timestamp("2026-05-07")    # test: 2025 to 2026-05-06

FINAL_MIN_GROUP_COUNT = 5
LOOKBACK_YEARS = 2

## Load and Merge Raw EIA Files

---

The raw files are still at balancing authority level, but the final forecasting target will be aggregated to region level.

In [4]:
keep_cols = [
    "Balancing Authority",
    "Data Date",
    "Hour Number",
    "Local Time at End of Hour",
    "UTC Time at End of Hour",
    "Demand (MW)",
    "Demand Forecast (MW)",
    "Region"
]

all_dfs = []

for dataset_path in dataset_paths:
    df = pd.read_csv(dataset_path)
    
    existing_cols = [col for col in keep_cols if col in df.columns]
    temp = df[existing_cols].copy()
    temp["source_file"] = Path(dataset_path).stem
    
    all_dfs.append(temp)

df_raw = pd.concat(all_dfs, ignore_index=True)

df_raw["timestamp_utc"] = pd.to_datetime(
    df_raw["UTC Time at End of Hour"],
    errors="coerce",
    utc=True
)

df_raw["data_date"] = pd.to_datetime(
    df_raw["Data Date"],
    errors="coerce"
)

df_raw["Demand (MW)"] = pd.to_numeric(
    df_raw["Demand (MW)"],
    errors="coerce"
)

df_raw["Demand Forecast (MW)"] = pd.to_numeric(
    df_raw["Demand Forecast (MW)"],
    errors="coerce"
)

df_raw["Hour Number"] = pd.to_numeric(
    df_raw["Hour Number"],
    errors="coerce"
)

df_raw = df_raw[
    (df_raw["data_date"] >= warmup_start_date) &
    (df_raw["data_date"] < test_end_date)
].copy()

df_raw = df_raw.sort_values(
    ["Region", "Balancing Authority", "timestamp_utc"]
).reset_index(drop=True)

print("Raw clipped shape:", df_raw.shape)
print("Local date range:", df_raw["data_date"].min(), "to", df_raw["data_date"].max())
print("UTC timestamp range:", df_raw["timestamp_utc"].min(), "to", df_raw["timestamp_utc"].max())
print("Regions:", sorted(df_raw["Region"].dropna().unique()))
print("Number of balancing authorities:", df_raw["Balancing Authority"].nunique())

df_raw.head()

Raw clipped shape: (6056162, 11)
Local date range: 2015-07-01 00:00:00 to 2026-05-06 00:00:00
UTC timestamp range: 2015-07-01 05:00:00+00:00 to 2026-05-07 07:00:00+00:00
Regions: ['CAL', 'CAR', 'CENT', 'FLA', 'MIDA', 'MIDW', 'NE', 'NW', 'NY', 'SE', 'SW', 'TEN', 'TEX']
Number of balancing authorities: 72


,Balancing Authority,Data Date,Hour Number,Local Time at End of Hour,UTC Time at End of Hour,Demand (MW),Demand Forecast (MW),Region,source_file,timestamp_utc,data_date
0,BANC,07/01/2015,1,07/01/2015 1:00:00 AM,07/01/2015 8:00:00 AM,2513.0,2226.0,CAL,EIA930_BALANCE_2015_Jul_Dec,2015-07-01 08:00:00+00:00,2015-07-01
1,BANC,07/01/2015,2,07/01/2015 2:00:00 AM,07/01/2015 9:00:00 AM,2275.0,2035.0,CAL,EIA930_BALANCE_2015_Jul_Dec,2015-07-01 09:00:00+00:00,2015-07-01
2,BANC,07/01/2015,3,07/01/2015 3:00:00 AM,07/01/2015 10:00:00 AM,2104.0,1897.0,CAL,EIA930_BALANCE_2015_Jul_Dec,2015-07-01 10:00:00+00:00,2015-07-01
3,BANC,07/01/2015,4,07/01/2015 4:00:00 AM,07/01/2015 11:00:00 AM,1988.0,1821.0,CAL,EIA930_BALANCE_2015_Jul_Dec,2015-07-01 11:00:00+00:00,2015-07-01
4,BANC,07/01/2015,5,07/01/2015 5:00:00 AM,07/01/2015 12:00:00 PM,1958.0,1811.0,CAL,EIA930_BALANCE_2015_Jul_Dec,2015-07-01 12:00:00+00:00,2015-07-01


## Check Balancing Authority Coverage per Region

---

This check confirms that each region contains multiple balancing authorities, so aggregation is required before region-level modeling.

In [5]:
ba_region_summary = (
    df_raw
    .groupby("Region")["Balancing Authority"]
    .nunique()
    .reset_index(name="num_balancing_authorities")
    .sort_values("num_balancing_authorities", ascending=False)
)

ba_region_summary

,Region,num_balancing_authorities
7,NW,24
3,FLA,10
10,SW,10
1,CAR,6
5,MIDW,6
0,CAL,5
9,SE,3
2,CENT,2
4,MIDA,2
6,NE,1


## Aggregate Raw Data to Region-Level Hourly Demand

---

Since the project target is regional demand, raw balancing authority rows are aggregated into one row per region and UTC timestamp.

In [6]:
df_region_raw = (
    df_raw
    .groupby(["Region", "timestamp_utc"], as_index=False)
    .agg(
        data_date=("data_date", "min"),
        region_demand_mw=("Demand (MW)", "sum"),
        region_demand_forecast_mw=("Demand Forecast (MW)", "sum"),
        num_ba=("Balancing Authority", "nunique"),
        num_rows=("Balancing Authority", "size"),
        num_available_ba=("Demand (MW)", lambda x: x.notna().sum()),
        num_missing_ba=("Demand (MW)", lambda x: x.isna().sum())
    )
)

# If all BA demand values are missing in a region-hour, the regional demand should be missing, not zero.
df_region_raw.loc[
    df_region_raw["num_available_ba"] == 0,
    "region_demand_mw"
] = np.nan

df_region_raw = df_region_raw.sort_values(
    ["Region", "timestamp_utc"]
).reset_index(drop=True)

print("Region-level raw shape:", df_region_raw.shape)
print("Max rows per Region + timestamp:",
      df_region_raw.groupby(["Region", "timestamp_utc"]).size().max())

df_region_raw.head()

Region-level raw shape: (1236457, 9)
Max rows per Region + timestamp: 1


,Region,timestamp_utc,data_date,region_demand_mw,region_demand_forecast_mw,num_ba,num_rows,num_available_ba,num_missing_ba
0,CAL,2015-07-01 08:00:00+00:00,2015-07-01,38210.0,35264.0,5,5,5,0
1,CAL,2015-07-01 09:00:00+00:00,2015-07-01,35171.0,32894.0,5,5,5,0
2,CAL,2015-07-01 10:00:00+00:00,2015-07-01,33243.0,31360.0,5,5,5,0
3,CAL,2015-07-01 11:00:00+00:00,2015-07-01,31955.0,30579.0,5,5,5,0
4,CAL,2015-07-01 12:00:00+00:00,2015-07-01,31199.0,30723.0,5,5,5,0


## Validate Region-Level Structure

In [7]:
region_structure_check = (
    df_region_raw
    .groupby("Region")
    .agg(
        min_timestamp=("timestamp_utc", "min"),
        max_timestamp=("timestamp_utc", "max"),
        n_rows=("timestamp_utc", "size"),
        n_unique_timestamps=("timestamp_utc", "nunique"),
        missing_region_demand=("region_demand_mw", lambda x: x.isna().sum()),
        missing_region_demand_pct=("region_demand_mw", lambda x: x.isna().mean() * 100),
        avg_num_ba=("num_ba", "mean")
    )
    .reset_index()
)

region_structure_check

,Region,min_timestamp,max_timestamp,n_rows,n_unique_timestamps,missing_region_demand,missing_region_demand_pct,avg_num_ba
0,CAL,2015-07-01 08:00:00+00:00,2026-05-07 07:00:00+00:00,95112,95112,24,0.025233,5.000000
1,CAR,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,24,0.025233,5.999748
2,CENT,2015-07-01 06:00:00+00:00,2026-05-07 05:00:00+00:00,95112,95112,24,0.025233,2.000000
3,FLA,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,24,0.025233,9.416866
4,MIDA,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,214,0.224998,1.315176
5,MIDW,2015-07-01 06:00:00+00:00,2026-05-07 05:00:00+00:00,95112,95112,24,0.025233,3.746404
6,NE,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,30,0.031542,1.000000
7,NW,2015-07-01 07:00:00+00:00,2026-05-07 07:00:00+00:00,95113,95113,24,0.025233,20.683103
8,NY,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,24,0.025233,1.000000
9,SE,2015-07-01 06:00:00+00:00,2026-05-07 05:00:00+00:00,95112,95112,71,0.074649,2.568509


## Create Complete Hourly Timeline per Region

---

This prevents incorrect lag features caused by missing timestamps.

In [8]:
def make_complete_hourly_region(group):
    region = group["Region"].iloc[0]
    
    group = group.sort_values("timestamp_utc").drop_duplicates("timestamp_utc")
    group = group.set_index("timestamp_utc")
    
    full_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="h",
        tz="UTC"
    )
    
    group = group.reindex(full_index)
    group.index.name = "timestamp_utc"
    group["Region"] = region
    
    return group.reset_index()


df_region_complete = (
    df_region_raw
    .groupby("Region", group_keys=False)
    .apply(make_complete_hourly_region)
    .reset_index(drop=True)
)

df_region_complete = df_region_complete.sort_values(
    ["Region", "timestamp_utc"]
).reset_index(drop=True)

print("Complete region-level shape:", df_region_complete.shape)
print("Missing region demand:", df_region_complete["region_demand_mw"].isna().sum())

df_region_complete.head()

C:\Users\surya\AppData\Local\Temp\ipykernel_8388\2691991330.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_complete_hourly_region)


Complete region-level shape: (1236457, 9)
Missing region demand: 1128


,timestamp_utc,Region,data_date,region_demand_mw,region_demand_forecast_mw,num_ba,num_rows,num_available_ba,num_missing_ba
0,2015-07-01 08:00:00+00:00,CAL,2015-07-01,38210.0,35264.0,5,5,5,0
1,2015-07-01 09:00:00+00:00,CAL,2015-07-01,35171.0,32894.0,5,5,5,0
2,2015-07-01 10:00:00+00:00,CAL,2015-07-01,33243.0,31360.0,5,5,5,0
3,2015-07-01 11:00:00+00:00,CAL,2015-07-01,31955.0,30579.0,5,5,5,0
4,2015-07-01 12:00:00+00:00,CAL,2015-07-01,31199.0,30723.0,5,5,5,0


## Add Time Features for Imputation

---

These features are used only for causal historical imputation. Final calendar features can be refined later.

In [9]:
df_region_complete["year"] = df_region_complete["timestamp_utc"].dt.year
df_region_complete["month"] = df_region_complete["timestamp_utc"].dt.month
df_region_complete["hour"] = df_region_complete["timestamp_utc"].dt.hour
df_region_complete["day_of_week"] = df_region_complete["timestamp_utc"].dt.dayofweek

df_region_complete.head()

,timestamp_utc,Region,data_date,region_demand_mw,region_demand_forecast_mw,num_ba,num_rows,num_available_ba,num_missing_ba,year,month,hour,day_of_week
0,2015-07-01 08:00:00+00:00,CAL,2015-07-01,38210.0,35264.0,5,5,5,0,2015,7,8,2
1,2015-07-01 09:00:00+00:00,CAL,2015-07-01,35171.0,32894.0,5,5,5,0,2015,7,9,2
2,2015-07-01 10:00:00+00:00,CAL,2015-07-01,33243.0,31360.0,5,5,5,0,2015,7,10,2
3,2015-07-01 11:00:00+00:00,CAL,2015-07-01,31955.0,30579.0,5,5,5,0,2015,7,11,2
4,2015-07-01 12:00:00+00:00,CAL,2015-07-01,31199.0,30723.0,5,5,5,0,2015,7,12,2


## Check negative and extreme demand values

In [10]:
warmup_start_date = pd.Timestamp("2015-07-01")
modeling_start_date = pd.Timestamp("2017-07-01")

train_end_date = pd.Timestamp("2024-01-01")
valid_end_date = pd.Timestamp("2025-01-01")
test_end_date = pd.Timestamp("2026-05-07")

df_region_complete["timestamp_utc"] = pd.to_datetime(df_region_complete["timestamp_utc"], utc=True)
df_region_complete["data_date"] = pd.to_datetime(df_region_complete["data_date"])

warmup_train_mask = (
    (df_region_complete["data_date"] >= warmup_start_date) &
    (df_region_complete["data_date"] < train_end_date)
)

valid_test_mask = (
    (df_region_complete["data_date"] >= train_end_date) &
    (df_region_complete["data_date"] < test_end_date)
)

In [11]:
target_col = "region_demand_mw"

print("Overall demand summary:")
display(df_region_complete[target_col].describe())

negative_rows = df_region_complete[df_region_complete[target_col] < 0].copy()

print("Number of negative demand rows:", len(negative_rows))

display(
    negative_rows[
        [
            "timestamp_utc",
            "data_date",
            "Region",
            target_col,
            "region_demand_forecast_mw",
            "num_ba",
            "num_available_ba",
            "num_missing_ba"
        ]
    ].sort_values(target_col).head(30)
)

Overall demand summary:


count    1.235329e+06
mean     3.866478e+04
std      2.405401e+06
min     -9.991332e+07
25%      1.816600e+04
50%      2.762600e+04
75%      4.113800e+04
max      2.147480e+09
Name: region_demand_mw, dtype: float64

Number of negative demand rows: 328


,timestamp_utc,data_date,Region,region_demand_mw,region_demand_forecast_mw,num_ba,num_available_ba,num_missing_ba
1063892,2017-07-06 01:00:00+00:00,2017-07-05,TEN,-99913317.0,23169.0,1,1,0
670144,2015-12-29 23:00:00+00:00,2015-12-29,NW,-31951234.0,43007.0,20,17,3
526955,2021-05-11 17:00:00+00:00,2021-05-11,MIDW,-16702550.0,76451.0,4,3,1
299288,2017-02-01 13:00:00+00:00,2017-02-01,FLA,-13413457.0,24399.0,10,10,0
670143,2015-12-29 22:00:00+00:00,2015-12-29,NW,-12919820.0,43192.0,20,17,3
1100040,2021-08-20 05:00:00+00:00,2021-08-19,TEN,-9946969.0,18174.0,1,1,0
33977,2019-05-17 01:00:00+00:00,2019-05-16,CAL,-2204469.0,30405.0,5,5,0
33684,2019-05-04 20:00:00+00:00,2019-05-04,CAL,-2177411.0,24670.0,5,5,0
33685,2019-05-04 21:00:00+00:00,2019-05-04,CAL,-2177080.0,24859.0,5,5,0
33686,2019-05-04 22:00:00+00:00,2019-05-04,CAL,-2176247.0,25431.0,5,5,0


In [12]:
negative_period_summary = pd.DataFrame([
    {
        "period": "warmup_train",
        "rows": warmup_train_mask.sum(),
        "negative_demand_rows": ((df_region_complete[target_col] < 0) & warmup_train_mask).sum()
    },
    {
        "period": "validation_test",
        "rows": valid_test_mask.sum(),
        "negative_demand_rows": ((df_region_complete[target_col] < 0) & valid_test_mask).sum()
    }
])

negative_period_summary

,period,rows,negative_demand_rows
0,warmup_train,969085,305
1,validation_test,267372,23


In [13]:
df_region_complete["is_negative_demand"] = (
    df_region_complete[target_col].notna() &
    (df_region_complete[target_col] < 0)
)

# Set negative values to missing.
df_region_complete.loc[df_region_complete["is_negative_demand"], target_col] = np.nan

print("Negative values converted to NaN:", df_region_complete["is_negative_demand"].sum())
print("Missing demand after negative cleanup:", df_region_complete[target_col].isna().sum())

Negative values converted to NaN: 328
Missing demand after negative cleanup: 1456


In [14]:
positive_demand = df_region_complete[df_region_complete[target_col] > 0][target_col]

q_summary = positive_demand.quantile([0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999])

display(q_summary)

extreme_threshold = positive_demand.quantile(0.999)

extreme_rows = df_region_complete[
    df_region_complete[target_col] > extreme_threshold
].copy()

print("Extreme threshold:", extreme_threshold)
print("Number of extreme high rows:", len(extreme_rows))

display(
    extreme_rows[
        [
            "timestamp_utc",
            "data_date",
            "Region",
            target_col,
            "region_demand_forecast_mw",
            "temperature_2m" if "temperature_2m" in extreme_rows.columns else "num_ba",
        ]
    ].head(30)
)

0.001        75.000
0.010      9025.000
0.050     11030.000
0.500     27638.000
0.950     90716.000
0.990    112984.000
0.999    135912.447
Name: region_demand_mw, dtype: float64

Extreme threshold: 135912.44700000016
Number of extreme high rows: 1235


,timestamp_utc,data_date,Region,region_demand_mw,region_demand_forecast_mw,num_ba
33052,2019-04-08 12:00:00+00:00,2019-04-08,CAL,1846001.0,23333.0,5
33078,2019-04-09 14:00:00+00:00,2019-04-09,CAL,1853155.0,28278.0,5
33157,2019-04-12 21:00:00+00:00,2019-04-12,CAL,1340285.0,26216.0,5
33424,2019-04-24 00:00:00+00:00,2019-04-23,CAL,313029.0,34763.0,5
33582,2019-04-30 14:00:00+00:00,2019-04-30,CAL,1887229.0,28095.0,5
33853,2019-05-11 21:00:00+00:00,2019-05-11,CAL,1922832.0,25544.0,5
33932,2019-05-15 04:00:00+00:00,2019-05-14,CAL,1869939.0,35290.0,5
33958,2019-05-16 06:00:00+00:00,2019-05-15,CAL,1870912.0,30272.0,5
34010,2019-05-18 10:00:00+00:00,2019-05-18,CAL,1870925.0,24144.0,5
34023,2019-05-18 23:00:00+00:00,2019-05-18,CAL,280600.0,25341.0,5


## Define Causal Region-Level Imputation Function

---

Missing regional demand is imputed using only past data from the same region, same hour, and same day of week within the previous two years.

In [15]:
def causal_impute_region_demand_2y_no_fallback(
    df,
    target_col="region_demand_mw",
    group_col="Region",
    time_col="timestamp_utc",
    min_group_count=5,
    lookback_years=2
):
    df = df.copy()
    df = df.sort_values([group_col, time_col]).reset_index(drop=True)

    df["target_region_demand_mw"] = df[target_col]
    df["is_target_imputed"] = 0
    df["imputation_method"] = "observed"

    result_parts = []

    for group_value, g in df.groupby(group_col):
        g = g.copy().sort_values(time_col).reset_index(drop=True)

        missing_indices = g.index[g["target_region_demand_mw"].isna()]

        for idx in missing_indices:
            current_time = g.loc[idx, time_col]
            current_hour = g.loc[idx, "hour"]
            current_dow = g.loc[idx, "day_of_week"]

            lookback_start = current_time - pd.DateOffset(years=lookback_years)

            hist_group = g[
                (g[time_col] < current_time) &
                (g[time_col] >= lookback_start) &
                (g[target_col].notna()) &
                (g["hour"] == current_hour) &
                (g["day_of_week"] == current_dow)
            ]

            if len(hist_group) >= min_group_count:
                g.loc[idx, "target_region_demand_mw"] = hist_group[target_col].mean()
                g.loc[idx, "is_target_imputed"] = 1
                g.loc[idx, "imputation_method"] = "region_dow_hour_prev2y"
            else:
                g.loc[idx, "imputation_method"] = "not_imputed"

        result_parts.append(g)

    out = pd.concat(result_parts, ignore_index=True)
    out = out.sort_values([group_col, time_col]).reset_index(drop=True)

    return out

## Compare Imputation Thresholds

---

Validation and test periods are not target-imputed for evaluation fairness.

In [16]:
threshold_results = []

for min_count in [5, 10, 15, 20, 30]:
    temp = causal_impute_region_demand_2y_no_fallback(
        df_region_complete,
        target_col="region_demand_mw",
        group_col="Region",
        time_col="timestamp_utc",
        min_group_count=min_count,
        lookback_years=LOOKBACK_YEARS
    )

    # Do not use target imputation in validation/test period.
    eval_mask = temp["data_date"] >= train_end_date
    
    temp.loc[eval_mask, "target_region_demand_mw"] = temp.loc[
        eval_mask, "region_demand_mw"
    ]
    
    temp.loc[eval_mask, "is_target_imputed"] = 0
    
    temp.loc[eval_mask, "imputation_method"] = np.where(
        temp.loc[eval_mask, "region_demand_mw"].notna(),
        "observed_eval_period",
        "missing_eval_period_not_imputed"
    )

    train_mask = (
        (temp["data_date"] >= modeling_start_date) &
        (temp["data_date"] < train_end_date)
    )

    valid_mask = (
        (temp["data_date"] >= train_end_date) &
        (temp["data_date"] < valid_end_date)
    )

    test_mask = (
        (temp["data_date"] >= valid_end_date) &
        (temp["data_date"] < test_end_date)
    )

    threshold_results.append({
        "min_group_count": min_count,
        "train_remaining_missing": temp.loc[train_mask, "target_region_demand_mw"].isna().sum(),
        "train_imputed_rows": temp.loc[train_mask, "is_target_imputed"].sum(),
        "valid_actual_missing": temp.loc[valid_mask, "region_demand_mw"].isna().sum(),
        "test_actual_missing": temp.loc[test_mask, "region_demand_mw"].isna().sum(),
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df

,min_group_count,train_remaining_missing,train_imputed_rows,valid_actual_missing,test_actual_missing
0,5,0,543,60,444
1,10,0,543,60,444
2,15,0,543,60,444
3,20,0,543,60,444
4,30,0,543,60,444


## Run Final Region-Level Imputation

---

Use the selected threshold from the comparison above. For simplicity and coverage, this notebook uses FINAL_MIN_GROUP_COUNT = 5.

In [17]:
df_region_imputed = causal_impute_region_demand_2y_no_fallback(
    df_region_complete,
    target_col="region_demand_mw",
    group_col="Region",
    time_col="timestamp_utc",
    min_group_count=FINAL_MIN_GROUP_COUNT,
    lookback_years=LOOKBACK_YEARS
)

# Do not impute target in validation/test period.
eval_mask = df_region_imputed["data_date"] >= train_end_date

df_region_imputed.loc[eval_mask, "target_region_demand_mw"] = df_region_imputed.loc[
    eval_mask, "region_demand_mw"
]

df_region_imputed.loc[eval_mask, "is_target_imputed"] = 0

df_region_imputed.loc[eval_mask, "imputation_method"] = np.where(
    df_region_imputed.loc[eval_mask, "region_demand_mw"].notna(),
    "observed_eval_period",
    "missing_eval_period_not_imputed"
)

print("Original missing region demand:", df_region_complete["region_demand_mw"].isna().sum())
print("After final imputation:", df_region_imputed["target_region_demand_mw"].isna().sum())
print("Imputed rows:", df_region_imputed["is_target_imputed"].sum())

df_region_imputed["imputation_method"].value_counts()

Original missing region demand: 1456
After final imputation: 504
Imputed rows: 952


imputation_method
observed                           968133
observed_eval_period               266868
region_dow_hour_prev2y                952
missing_eval_period_not_imputed       504
Name: count, dtype: int64

## Summarize Imputation by Region

In [18]:
imputation_summary_region = (
    df_region_imputed
    .groupby("Region")
    .agg(
        total_rows=("target_region_demand_mw", "size"),
        original_missing=("region_demand_mw", lambda x: x.isna().sum()),
        imputed_rows=("is_target_imputed", "sum"),
        remaining_missing=("target_region_demand_mw", lambda x: x.isna().sum())
    )
    .reset_index()
)

imputation_summary_region["original_missing_pct"] = (
    imputation_summary_region["original_missing"] /
    imputation_summary_region["total_rows"] * 100
)

imputation_summary_region["imputed_pct"] = (
    imputation_summary_region["imputed_rows"] /
    imputation_summary_region["total_rows"] * 100
)

imputation_summary_region["remaining_missing_pct"] = (
    imputation_summary_region["remaining_missing"] /
    imputation_summary_region["total_rows"] * 100
)

imputation_summary_region.sort_values("imputed_pct", ascending=False)

,Region,total_rows,original_missing,imputed_rows,remaining_missing,original_missing_pct,imputed_pct,remaining_missing_pct
11,TEN,95112,313,289,24,0.329086,0.303852,0.025233
12,TEX,95112,312,240,72,0.328034,0.252334,0.075700
0,CAL,95112,245,221,24,0.257591,0.232358,0.025233
4,MIDA,95112,214,96,118,0.224998,0.100934,0.124064
7,NW,95113,75,49,26,0.078854,0.051518,0.027336
9,SE,95112,72,24,48,0.075700,0.025233,0.050467
3,FLA,95112,46,16,30,0.048364,0.016822,0.031542
10,SW,95112,49,10,39,0.051518,0.010514,0.041004
6,NE,95112,30,3,27,0.031542,0.003154,0.028388
2,CENT,95112,26,2,24,0.027336,0.002103,0.025233


## Split Warm-Up, Train, Validation, and Test Sets

---

Training can use imputed targets. Validation and test sets use actual regional demand only.

In [19]:
df_warmup_region = df_region_imputed[
    (df_region_imputed["data_date"] >= warmup_start_date) &
    (df_region_imputed["data_date"] < modeling_start_date)
].copy()

df_train_region_full = df_region_imputed[
    (df_region_imputed["data_date"] >= modeling_start_date) &
    (df_region_imputed["data_date"] < train_end_date)
].copy()

df_valid_region_full = df_region_imputed[
    (df_region_imputed["data_date"] >= train_end_date) &
    (df_region_imputed["data_date"] < valid_end_date)
].copy()

df_test_region_full = df_region_imputed[
    (df_region_imputed["data_date"] >= valid_end_date) &
    (df_region_imputed["data_date"] < test_end_date)
].copy()

# Train can use imputed target.
df_train_region = df_train_region_full.dropna(
    subset=["target_region_demand_mw"]
).copy()

# Validation/test evaluate actual target only.
df_valid_region = df_valid_region_full.dropna(
    subset=["region_demand_mw"]
).copy()

df_test_region = df_test_region_full.dropna(
    subset=["region_demand_mw"]
).copy()

print("Warm-up region:", df_warmup_region.shape)
print("Train full:", df_train_region_full.shape, "| Train usable:", df_train_region.shape)
print("Validation full:", df_valid_region_full.shape, "| Validation actual only:", df_valid_region.shape)
print("Test full:", df_test_region_full.shape, "| Test actual only:", df_test_region.shape)

Warm-up region: (228073, 17)
Train full: (741012, 17) | Train usable: (741012, 17)
Validation full: (114192, 17) | Validation actual only: (114132, 17)
Test full: (153180, 17) | Test actual only: (152736, 17)


## Validate Final Region-Level Dataset

In [20]:
final_structure_check = (
    df_region_imputed
    .groupby(["Region", "timestamp_utc"])
    .size()
    .reset_index(name="count")
)

print("Max rows per Region + timestamp:", final_structure_check["count"].max())

final_region_summary = (
    df_region_imputed
    .groupby("Region")
    .agg(
        min_timestamp=("timestamp_utc", "min"),
        max_timestamp=("timestamp_utc", "max"),
        n_rows=("timestamp_utc", "size"),
        n_unique_timestamps=("timestamp_utc", "nunique"),
        missing_actual=("region_demand_mw", lambda x: x.isna().sum()),
        missing_target=("target_region_demand_mw", lambda x: x.isna().sum()),
        imputed_rows=("is_target_imputed", "sum")
    )
    .reset_index()
)

final_region_summary

Max rows per Region + timestamp: 1


,Region,min_timestamp,max_timestamp,n_rows,n_unique_timestamps,missing_actual,missing_target,imputed_rows
0,CAL,2015-07-01 08:00:00+00:00,2026-05-07 07:00:00+00:00,95112,95112,245,24,221
1,CAR,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,24,24,0
2,CENT,2015-07-01 06:00:00+00:00,2026-05-07 05:00:00+00:00,95112,95112,26,24,2
3,FLA,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,46,30,16
4,MIDA,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,214,118,96
5,MIDW,2015-07-01 06:00:00+00:00,2026-05-07 05:00:00+00:00,95112,95112,26,24,2
6,NE,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,30,27,3
7,NW,2015-07-01 07:00:00+00:00,2026-05-07 07:00:00+00:00,95113,95113,75,26,49
8,NY,2015-07-01 05:00:00+00:00,2026-05-07 04:00:00+00:00,95112,95112,24,24,0
9,SE,2015-07-01 06:00:00+00:00,2026-05-07 05:00:00+00:00,95112,95112,72,48,24


## Save Reproducibility Files

In [21]:
df_region_raw.to_csv(
    f"{OUTPUT_DIR}/region_raw_aggregated_before_imputation.csv",
    index=False
)

df_region_complete.to_csv(
    f"{OUTPUT_DIR}/region_complete_hourly_before_imputation.csv",
    index=False
)

df_region_imputed.to_csv(
    f"{OUTPUT_DIR}/region_hourly_after_causal_imputation.csv",
    index=False
)

df_warmup_region.to_csv(
    f"{OUTPUT_DIR}/region_warmup_2015H2_2017H1.csv",
    index=False
)

df_train_region.to_csv(
    f"{OUTPUT_DIR}/region_train_usable_2017H2_2023H2.csv",
    index=False
)

df_valid_region.to_csv(
    f"{OUTPUT_DIR}/region_valid_actual_only_2024.csv",
    index=False
)

df_test_region.to_csv(
    f"{OUTPUT_DIR}/region_test_actual_only_2025_2026May.csv",
    index=False
)

threshold_results_df.to_csv(
    f"{OUTPUT_DIR}/region_imputation_threshold_comparison.csv",
    index=False
)

imputation_summary_region.to_csv(
    f"{OUTPUT_DIR}/region_imputation_summary.csv",
    index=False
)

print("Saved files to:", OUTPUT_DIR)

Saved files to: electricityEDA_outputs/reproducibility


## Save Preprocessing Metadata

In [22]:
metadata = pd.DataFrame([
    {"item": "raw_unit", "value": "Balancing Authority hourly rows"},
    {"item": "final_unit", "value": "Region hourly total demand"},
    {"item": "warmup_start_date", "value": str(warmup_start_date)},
    {"item": "modeling_start_date", "value": str(modeling_start_date)},
    {"item": "train_end_date_exclusive", "value": str(train_end_date)},
    {"item": "valid_end_date_exclusive", "value": str(valid_end_date)},
    {"item": "test_end_date_exclusive", "value": str(test_end_date)},
    {"item": "final_min_group_count", "value": FINAL_MIN_GROUP_COUNT},
    {"item": "lookback_years", "value": LOOKBACK_YEARS},
    {"item": "aggregation_rule", "value": "sum Demand (MW) by Region and timestamp_utc"},
    {"item": "imputation_rule", "value": "same Region + day_of_week + hour, previous 2 years only"},
    {"item": "train_target", "value": "target_region_demand_mw"},
    {"item": "validation_test_target", "value": "region_demand_mw actual only"},
    {"item": "demand_forecast_usage", "value": "benchmark only, not used as model feature"}
])

metadata.to_csv(
    f"{OUTPUT_DIR}/preprocessing_metadata.csv",
    index=False
)

metadata

,item,value
0,raw_unit,Balancing Authority hourly rows
1,final_unit,Region hourly total demand
2,warmup_start_date,2015-07-01 00:00:00
3,modeling_start_date,2017-07-01 00:00:00
4,train_end_date_exclusive,2024-01-01 00:00:00
5,valid_end_date_exclusive,2025-01-01 00:00:00
6,test_end_date_exclusive,2026-05-07 00:00:00
7,final_min_group_count,5
8,lookback_years,2
9,aggregation_rule,sum Demand (MW) by Region and timestamp_utc
